[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

Implement **cross-entropy loss** from scratch.

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

### Signature
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) float, targets: (B,) long indices
    # Returns: scalar loss (mean over batch)
```

### Rules
- Do NOT use `F.cross_entropy` or `nn.CrossEntropyLoss`
- Must be numerically stable (use logsumexp trick)

In [12]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [14]:
import torch

In [19]:
# ✏️ YOUR IMPLEMENTATION HERE

'''
logsumexp trick: log(Σ exp(x_j)) = max + log(Σ exp(x_j - max))

log(Σ exp(x_j))

  = log(Σ exp(x_j - m + m))          # 加减 m（m = max）

  = log(Σ exp(x_j - m) · exp(m))     # 指数拆开

  = log(exp(m) · Σ exp(x_j - m))     # 提出常数 exp(m)

  = log(exp(m)) + log(Σ exp(x_j - m)) # log(ab) = log(a) + log(b)

  = m + log(Σ exp(x_j - m))          # 完成

'''
def cross_entropy_loss(logits, targets):
    max_logits = logits.max(dim=-1, keepdim=True).values
    
    '''
    max_logits              # (B, 1) → tensor([[3.0], [5.0]])
    max_logits.squeeze(-1)  # (B,)   → tensor([3.0, 5.0])
    '''
    
    log_sum_exp = max_logits.squeeze(-1) + torch.log(torch.exp(logits - max_logits).sum(dim=-1))
    
    # 取出每个样本正确类别的 logit, 
    correct_logits = logits[torch.arange(logits.shape[0]), targets] # (B, 1)
    
    return (log_sum_exp - correct_logits).mean()
    
    # log_probs = logits - logsumexp(...)

In [20]:
# 🧪 Debug
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

Loss: tensor(2.4959)
Ref:  tensor(2.4959)


In [9]:
# ✅ SUBMIT
from torch_judge import check
check('cross_entropy')


🧪 Testing: Cross-Entropy Loss (Easy)
──────────────────────────────────────────────────
  ❌ [1/4] Matches F.cross_entropy
     Mismatch: -11.5398 vs 2.9977
  ❌ [2/4] Numerical stability
     Inf with large logits
  ✅ [3/4] Scalar output (0.0ms)
  ✅ [4/4] Gradient flow (17.2ms)
──────────────────────────────────────────────────
  📊 2/4 tests passed.
  Keep going! Use hint("cross_entropy") if you're stuck.

